In [1]:
using ITensors
using ITensorMPS: op
using KrylovKit: schursolve, Arnoldi
using LinearAlgebra

In [2]:
struct iMPS2
    Γa::ITensor
    λa::ITensor
    Γb::ITensor
    λb::ITensor
    iMPS2(Γa::ITensor, λa::ITensor, Γb::ITensor, λb::ITensor) = new(Γa, λa, Γb, λb)
end

function iMPS2(Γa::Array{ComplexF64, 3}, λa::Array{ComplexF64, 2}, Γb::Array{ComplexF64, 3}, λb::Array{ComplexF64, 2})
    l02 = Index(size(λb, 2), tags="Link,n=0,l=2")
    l11 = Index(size(λa, 1), tags="Link,n=1,l=1")
    l12 = Index(size(λa, 2), tags="Link,n=1,l=2")
    l21 = Index(size(λb, 1), tags="Link,n=2,l=1")
    l22 = Index(size(λb, 2), tags="Link,n=2,l=2")

    s1 = Index(size(Γa, 2), tags="Site,n=1")
    s2 = Index(size(Γb, 2), tags="Site,n=2")

    Γa_itensor = ITensor(Γa, (l02, s1, l11))
    λa_itensor = ITensor(λa, (l11, l12))
    Γb_itensor = ITensor(Γb, (l12, s2, l21))
    λb_itensor = ITensor(λb, (l21, l22))
    return iMPS2(Γa_itensor, λa_itensor, Γb_itensor, λb_itensor)
end

function reset_indices(ψ::iMPS2)
    l02, l11, l12, l21, l22 = linkinds(ψ)
    s1, s2 = siteinds(ψ)

    l02_new = replacetags(l02, tags(l02) => "Link,n=0,l=2")
    l11_new = replacetags(l11, tags(l11) => "Link,n=1,l=1")
    l12_new = replacetags(l12, tags(l12) => "Link,n=1,l=2")
    l21_new = replacetags(l21, tags(l21) => "Link,n=2,l=1")
    l22_new = replacetags(l22, tags(l22) => "Link,n=2,l=2")
    s1_new = replacetags(s1, tags(s1) => "Site,n=1")
    s2_new = replacetags(s2, tags(s2) => "Site,n=2")

    Γa = setprime(replaceinds(ψ.Γa, (l02 => l02_new, s1 => s1_new, l11 => l11_new)), 0)
    λa = setprime(replaceinds(ψ.λa, (l11 => l11_new, l12 => l12_new)), 0)
    Γb = setprime(replaceinds(ψ.Γb, (l12 => l12_new, s2 => s2_new, l21 => l21_new)), 0)
    λb = setprime(replaceinds(ψ.λb, (l21 => l21_new, l22 => l22_new)), 0)
    return iMPS2(Γa, λa, Γb, λb)
end

maxbonddim(ψ::iMPS2) = maximum([dim(ind) for ind in linkinds(ψ)])

maxbonddim (generic function with 1 method)

In [19]:
function eigenvec(A::ITensor, in_leg::Index{Int}, out_leg::Index{Int}; tol=1e-8, eager=true, krylovdim=40)
    Adag = dag(prime(A; tags="Link"))
    W = delta(in_leg, out_leg) * delta(prime(in_leg), prime(out_leg))
    
    map(X::ITensor) = W * (A * (Adag * X))

    v0 = randomITensor(in_leg, prime(in_leg))

    arn = Arnoldi(; tol, eager, krylovdim)

    TT, v, μ, info = schursolve(map, v0, 1, :LM, arn)
    μ = μ[1]
    V = v[1]

    if info.converged == 0
        @warn "map not converged after $(info.numiter) iterations"
    end
    if size(TT, 2) > 1 && TT[2, 1] != 0
        @warn "Non-unique largest eigenvector of map found"
    end

    return V, μ
end

evolve_AB(ψ::iMPS2, M::Matrix; kwargs...) = evolve(ψ, M; kwargs...)
evolve_BA(ψ::iMPS2, M::Matrix; kwargs...) = shift_cell(evolve(shift_cell(ψ), M; kwargs...))

function evolve(ψ::iMPS2, M::Matrix; maxdim=128, cutoff=1e-8, tol=1e-8, eager=true, krylovdim=40)
    Γa, λa, Γb, λb = ψ.Γa, ψ.λa, ψ.Γb, ψ.λb
    l02, l11, l12, l21, l22 = linkinds(ψ)
    s1, s2 = siteinds(ψ)
    l01 = Index(dim(l02), tags="Link,n=0,l=1")

    gate = op(M, [s1, s2])
    Φ = (Γa * λa * Γb) * gate
    Φ = replaceinds(Φ, prime(s1) => s1, prime(s2) => s2)

    Θ1 = Φ * λb
    VR, η = eigenvec(Θ1, l22, l02; tol=tol, eager=eager, krylovdim=krylovdim)

    Θ2 = replaceinds(λb, l21 => l01, l22 => l02) * Φ
    VL, τ = eigenvec(Θ2, l01, l21; tol=tol, eager=eager, krylovdim=krylovdim)
    
    Θ = Θ2 * λb

    X = decompose(VR, l22, prime(l22))
    Y = decompose(VL, prime(l01), l01) # actually Y this time (not Y^T)

    # are these index reuses safe? (next 4 lines)
    U, λb_new, V, _, u, v = svd(ITensor(transpose(Y), prime(l01, 2), l01) * ITensor(X, l01, prime(l01)), prime(l01, 2); maxdim=maxdim, cutoff=cutoff)
    λb_new /= norm(λb_new)

    λBVXinv = λb_new * V * ITensor(inv(X), prime(l01), l01)
    YTinvUλB = ITensor(inv(transpose(Y)), l22, prime(l22)) * replaceinds(U, prime(l01, 2) => prime(l22)) * λb_new

    Σ = λBVXinv * Θ * YTinvUλB
    # return Σ

    P, λa_new, Q, _, a, b = svd(Σ, u, s1; maxdim=maxdim, cutoff=cutoff)
    λa_new /= norm(λa_new)

    λb_inv_mat = diagm(diag(Matrix(λb_new, u, v).^(-1)))

    Γa_new = ITensor(λb_inv_mat, prime(u), u) * P
    Γb_new = Q * ITensor(λb_inv_mat, v, prime(v))
    λb_new = replaceinds(λb_new, v => prime(v))

    # println(Γa_new)
    # println(Γb_new)
    # println(λa_new)
    # println(λb_new)

    # Γa_new = ITensor(λb_inv_mat, l02, u) * replaceinds(P, a => l11)
    # Γb_new = replaceinds(Q, b => l12) * ITensor(λb_inv_mat, v, l21)
    # λa_new = replaceinds(λa_new, a => l11, b => l12)
    # λb_new = replaceinds(λb_new, u => l21, v => l22)
    return reset_indices(iMPS2(Γa_new, λa_new, Γb_new, λb_new))
end

function decompose(M::ITensor, in_leg::Index{Int}, out_leg::Index{Int})
    U, S, V, _, u, v = svd(M, in_leg)

    X = Matrix(U, in_leg, u) * Diagonal(sqrt.(diag(Matrix(S, u, v))))

    return X
end

function shift_cell(ψ::iMPS2)
    Γa, λa, Γb, λb = ψ.Γa, ψ.λa, ψ.Γb, ψ.λb

    l02, l11, l12, l21, l22 = linkinds(ψ)
    s1, s2 = siteinds(ψ) 

    l02_new = replacetags(l12, tags(l12) => tags(l02))
    l11_new = replacetags(l21, tags(l21) => tags(l11))
    l12_new = replacetags(l22, tags(l22) => tags(l12))
    l21_new = replacetags(l11, tags(l11) => tags(l21))
    l22_new = replacetags(l12, tags(l12) => tags(l22))

    s1_new = replacetags(s2, tags(s2) => tags(s1))
    s2_new = replacetags(s1, tags(s1) => tags(s2))

    Γa_new = replaceinds(Γb, l12 => l02_new, s2 => s1_new, l21 => l11_new)
    λa_new = replaceinds(λb, l21 => l11_new, l22 => l12_new)
    Γb_new = replaceinds(Γa, l02 => l12_new, s1 => s2_new, l11 => l21_new)
    λb_new = replaceinds(λa, l11 => l21_new, l12 => l22_new)

    shifted_ψ = iMPS2(Γa_new, λa_new, Γb_new, λb_new)

    return shifted_ψ
end

function linkinds(ψ::iMPS2)
    Γa, λa, Γb, λb = ψ.Γa, ψ.λa, ψ.Γb, ψ.λb

    l02 = only(setdiff(inds(Γa; tags="Link"), inds(λa)))
    l11 = commonind(Γa, λa)
    l12 = commonind(λa, Γb)
    l21 = commonind(Γb, λb)
    l22 = only(setdiff(inds(λb), inds(Γb)))

    return l02, l11, l12, l21, l22
end

function siteinds(ψ::iMPS2)
    Γa, λa, Γb, λb = ψ.Γa, ψ.λa, ψ.Γb, ψ.λb

    s1 = only(inds(Γa; tags="Site"))
    s2 = only(inds(Γb; tags="Site"))

    return s1, s2
end

function dephasing_gate(M::Matrix, p::Float64)
    return (1-p) * I + p * M
end

dephasing_gate (generic function with 1 method)

In [ ]:
Γa = ComplexF64.(reshape([1.0, 0.0], 1, 2, 1))
λa = ComplexF64.(reshape([1.0], 1, 1))
Γb = ComplexF64.(reshape([0.0, 1.0], 1, 2, 1))
λb = ComplexF64.(reshape([1.0], 1, 1))

ψ = iMPS2(Γa, λa, Γb, λb)

SWAP = [1 0 0 0
        0 0 1 0
        0 1 0 0
        0 0 0 1]



p = 0.1

for _ in 1:100
    ψ = evolve_AB(ψ, dephasing_gate(SWAP, p); cutoff=1e-8, tol=1e-8)
    println("Max dim: ", maxbonddim(ψ))
    ψ = evolve_BA(ψ, dephasing_gate(SWAP, p); cutoff=1e-8, tol=1e-8)
    println("Max dim: ", maxbonddim(ψ))
end

In [ ]:
Γa = ComplexF64.(reshape([1.0, 0.0], 1, 2, 1))
λa = ComplexF64.(reshape([1.0], 1, 1))
Γb = ComplexF64.(reshape([0.0, 1.0], 1, 2, 1))
λb = ComplexF64.(reshape([1.0], 1, 1))

ψ = iMPS2(Γa, λa, Γb, λb)

SWAP = [1 0 0 0
        0 0 1 0
        0 1 0 0
        0 0 0 1]

# println("Max dim: ", maxbonddim(ψ))

p = 0.1

# ψ0 = deepcopy(ψ)
# l02_0, l11_0, l12_0, l21_0, l22_0 = linkinds(ψ0)
# s1_0, s2_0 = siteinds(ψ0)

# ψ = evolve_AB(ψ, dephasing_gate(SWAP, p); cutoff=1e-16, tol=1e-16)

# l02, l11, l12, l21, l22 = linkinds(ψ)
# s1, s2 = siteinds(ψ)

# A1 = Array(ψ.Γa * ψ.λa * ψ.Γb * ψ.λb, l02, s1, s2, l22)

# M = M = dephasing_gate(SWAP, p)
# A2 = Array(ψ0.Γa * ψ0.λa * ψ0.Γb * ψ0.λb * op(M, siteinds(ψ0)...), l02_0, prime(s1_0), prime(s2_0), l22_0)
# A2 

for _ in 1:100
    ψ = evolve_AB(ψ, dephasing_gate(SWAP, p); cutoff=1e-8, tol=1e-8)
    println("Max dim: ", maxbonddim(ψ))
    ψ = evolve_BA(ψ, dephasing_gate(SWAP, p); cutoff=1e-8, tol=1e-8)
    println("Max dim: ", maxbonddim(ψ))
end

Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2
Max dim: 2

In [49]:
Array(ψ.Γa*ψ.λa*ψ.Γb*ψ.λb, inds(ψ.Γa*ψ.λa*ψ.Γb*ψ.λb)...)

2×1×2×1 Array{ComplexF64, 4}:
[:, :, 1, 1] =
 -0.4999241624552709 + 0.008679021684149496im
   0.499999946927154 + 1.4597267075813947e-12im

[:, :, 2, 1] =
   0.5000000530723325 + 1.459728049215931e-12im
 -0.49992517505514694 - 0.008679039266488315im

In [44]:
A1 .^ 2

1×2×2×1 Array{ComplexF64, 4}:
[:, :, 1, 1] =
 0.0+0.0im  0.0121951+0.0im

[:, :, 2, 1] =
 0.987805+0.0im  0.0+0.0im

In [24]:
M = dephasing_gate(SWAP, 1.0)
tol = 1e-16
maxdim = 128
cutoff = 1e-16
eager = true
krylovdim = 40

Γa, λa, Γb, λb = ψ.Γa, ψ.λa, ψ.Γb, ψ.λb
l02, l11, l12, l21, l22 = linkinds(ψ)
s1, s2 = siteinds(ψ)
l01 = Index(dim(l02), tags="Link,n=0,l=1")

gate = op(M, [s1, s2])
Φ = (Γa * λa * Γb) * gate
Φ = replaceinds(Φ, prime(s1) => s1, prime(s2) => s2)

Θ1 = Φ * λb
VR, η = eigenvec(Θ1, l22, l02; tol=tol, eager=eager, krylovdim=krylovdim)

Θ2 = replaceinds(λb, l21 => l01, l22 => l02) * Φ
VL, τ = eigenvec(Θ2, l01, l21; tol=tol, eager=eager, krylovdim=krylovdim)

Θ = Θ2 * λb

X = decompose(VR, l22, prime(l22))
Y = decompose(VL, prime(l01), l01) # actually Y this time (not Y^T)

# are these index reuses safe? (next 4 lines)
U, λb_new, V, _, u, v = svd(ITensor(transpose(Y), prime(l01, 2), l01) * ITensor(X, l01, prime(l01)), prime(l01, 2); maxdim=maxdim, cutoff=cutoff)
λb_new /= norm(λb_new)

λBVXinv = λb_new * V * ITensor(inv(X), prime(l01), l01)
YTinvUλB = ITensor(inv(transpose(Y)), l22, prime(l22)) * replaceinds(U, prime(l01, 2) => prime(l22)) * λb_new

Σ = λBVXinv * Θ * YTinvUλB
# # return Σ

# P, λa_new, Q, _, a, b = svd(Σ, u, s1; maxdim=maxdim, cutoff=cutoff)
# λa_new /= sum(λa_new)

# λb_inv_mat = diagm(diag(Matrix(λb_new, u, v).^(-1)))

# Γa_new = ITensor(λb_inv_mat, prime(u), u) * P
# Γb_new = Q * ITensor(λb_inv_mat, v, prime(v))
# λb_new = replaceinds(λb_new, v => prime(v))

# # println(Γa_new)
# # println(Γb_new)
# # println(λa_new)
# # println(λb_new)

# # Γa_new = ITensor(λb_inv_mat, l02, u) * replaceinds(P, a => l11)
# # Γb_new = replaceinds(Q, b => l12) * ITensor(λb_inv_mat, v, l21)
# # λa_new = replaceinds(λa_new, a => l11, b => l12)
# # λb_new = replaceinds(λb_new, u => l21, v => l22)
# return reset_indices(iMPS2(Γa_new, λa_new, Γb_new, λb_new))

ITensor ord=4 (dim=4|id=110|"Link,u") (dim=2|id=328|"Site,n=1") (dim=2|id=734|"Site,n=2") (dim=4|id=677|"Link,v")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

In [18]:
U, S, V, spec = svd(Σ, inds(Σ)[1:2])
norm(S)

0.5382982494978241

In [248]:
maxdim(ψ.Γa)

2

ITensors.TruncSVD(ITensor ord=3
Dim 1: (dim=1|id=705|"Link,u")
Dim 2: (dim=2|id=429|"Site,n=1")
Dim 3: (dim=2|id=805|"Link,u")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 1×2×2
[:, :, 1] =
 -1.0 + 0.0im  0.0 + 0.0im

[:, :, 2] =
 0.0 + 0.0im  -1.0 + 0.0im
, ITensor ord=2
Dim 1: (dim=2|id=805|"Link,u")
Dim 2: (dim=2|id=491|"Link,v")
NDTensors.Diag{Float64, Vector{Float64}}
 2×2
 0.9  0.0
 0.0  0.1
, ITensor ord=3
Dim 1: (dim=2|id=125|"Site,n=2")
Dim 2: (dim=1|id=501|"Link,v")
Dim 3: (dim=2|id=491|"Link,v")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2×1×2
[:, :, 1] =
 -0.0 + 0.0im
 -1.0 + 0.0im

[:, :, 2] =
 -1.0 + 0.0im
 -0.0 + 0.0im
, Spectrum{Vector{Float64}, Float64}([0.81, 0.010000000000000002], 0.0), (dim=2|id=805|"Link,u"), (dim=2|id=491|"Link,v"))

In [ ]:
Γa = ComplexF64.(reshape([1.0, 0.0], 1, 2, 1))
λa = ComplexF64.(reshape([1.0], 1, 1))
Γb = ComplexF64.(reshape([0.0, 1.0], 1, 2, 1))
λb = ComplexF64.(reshape([1.0], 1, 1))

ψ = iMPS2(Γa, λa, Γb, λb)


In [223]:
left_ind = only(setdiff(inds(ψ.Γa; tags="Link"), inds(ψ.λa)))
right_ind = only(commoninds(ψ.λa, ψ.Γb))
T = transfer_matrix(ψ.Γa * ψ.λa) * delta(right_ind, prime(right_ind))
Matrix(T, left_ind, prime(left_ind))

1×1 Matrix{ComplexF64}:
 1.0 + 0.0im

In [225]:
left_ind = only(commoninds(ψ.λa, ψ.Γb))
right_ind = only(setdiff(inds(ψ.λb), inds(ψ.Γb)))

T = transfer_matrix(ψ.Γb * ψ.λb) * delta(right_ind, prime(right_ind))
Matrix(T, left_ind, prime(left_ind))

1×1 Matrix{ComplexF64}:
 1.0 + 0.0im

In [229]:
left_ind = only(commoninds(ψ.Γa, ψ.λa))
right_ind = only(commoninds(ψ.Γb, ψ.λb))

T = transfer_matrix(ψ.λa * ψ.Γb) * delta(left_ind, prime(left_ind))
Matrix(T, right_ind, prime(right_ind))

1×1 Matrix{ComplexF64}:
 1.0 + 0.0im

In [230]:
l02, l11, l12, l21, l22 = linkinds(ψ)

T = transfer_matrix(ψ.λb * ψ.Γa * delta(l02, l22)) * delta(l21, prime(l21))
Matrix(T, l11, prime(l11))

1×1 Matrix{ComplexF64}:
 1.0 + 0.0im

(dim=1|id=208|"Link,l=2,n=1")

In [207]:
ψ.Γa

ITensor ord=3 (dim=2|id=633|"Site,n=1") (dim=4|id=944|"Link,l=2,n=0") (dim=4|id=888|"Link,l=1,n=1")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

In [194]:
setprime(ψ.Γa, 0)

ITensor ord=3 (dim=1|id=92|"Link,l=2,n=0") (dim=2|id=716|"Site,n=1") (dim=2|id=939|"Link,l=1,n=1")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

In [183]:
A = Index(4, "A")
prime(A,0)

(dim=4|id=609|"A")

In [108]:
prime(Index(4, "A"), 2)

(dim=4|id=798|"A")''

In [116]:
i = Index(4, "A")
j = Index(5, "B")

(dim=5|id=178|"B")

In [119]:
replacetags(i, tags(i) => tags(j))

(dim=4|id=43|"B")

In [97]:
struct iMPS2
    Γa::ITensor
    λa::ITensor
    Γb::ITensor
    λb::ITensor
    iMPS2(Γa::ITensor, λa::ITensor, Γb::ITensor, λb::ITensor) = new(Γa, λa, Γb, λb)
end

function iMPS2(Γa::Array{ComplexF64, 3}, λa::Array{ComplexF64, 2}, Γb::Array{ComplexF64, 3}, λb::Array{ComplexF64, 2})
    l02 = Index(size(λb, 2), tags="Link,n=0,l=2")
    l11 = Index(size(λa, 1), tags="Link,n=1,l=1")
    l12 = Index(size(λa, 2), tags="Link,n=1,l=2")
    l21 = Index(size(λb, 1), tags="Link,n=2,l=1")
    l22 = Index(size(λb, 2), tags="Link,n=2,l=2")

    s1 = Index(size(Γa, 2), tags="Site,n=1")
    s2 = Index(size(Γb, 2), tags="Site,n=2")

    Γa_itensor = ITensor(Γa, (l02, s1, l11))
    λa_itensor = ITensor(λa, (l11, l12))
    Γb_itensor = ITensor(Γb, (l12, s2, l21))
    λb_itensor = ITensor(λb, (l21, l22))
    return iMPS2(Γa_itensor, λa_itensor, Γb_itensor, λb_itensor)
end

transfer_matrix(tensor::ITensor) = tensor * dag(prime(tensor; tags="Link"))

function canonicalize(ψ::iMPS2; maxdim=128, cutoff=1e-8, tol=1e-8, eager=true, krylovdim=40)
    Γ = ψ.Γa * ψ.λa * ψ.Γb
    λ = ψ.λb

    Γ, λ = canonicalize(Γ, λ; maxdim=maxdim, cutoff=cutoff, tol=tol, eager=eager, krylovdim=krylovdim)
    Γa, λa, Γb, λb = split_canonical(Γ, λ, only(commoninds(ψ.Γa, ψ.λa)), only(commoninds(ψ.λa, ψ.Γb)); maxdim=maxdim, cutoff=cutoff)

    return iMPS2(Γa, λa, Γb, λb)
end

function split_canonical(Γ::ITensor, λb::ITensor, l11::Index{Int}, l12::Index{Int}; maxdim=128, cutoff=1e-8)
    l02 = only(inds(Γ; tags="Link,n=0,l=2"))
    s1 = only(inds(Γ; tags="Site,n=1"))
    s2 = only(inds(Γ; tags="Site,n=2"))
    l21 = only(inds(λb; tags="Link,n=2,l=1"))
    l22 = only(inds(λb; tags="Link,n=2,l=2"))

    M = replaceinds(λb, l21 => prime(l02), l22 => l02) * Γ * λb
    P, λa, Q, _, u, v = svd(M, prime(l02), s1; maxdim=maxdim, cutoff=cutoff)

    λB_inv_mat = diagm(diag(Matrix(λb, l21, l22).^(-1)))

    Γa = ITensor(λB_inv_mat, l02, prime(l02)) * replaceinds(P, u => l11)
    Γb = replaceinds(Q, v => prime(l12)) * ITensor(λB_inv_mat, prime(l22), l22)

    λa = replaceinds(λa, u => l11, v => l12)

    return Γa, λa, Γb, λb
end

function canonicalize(Γ::ITensor, λ::ITensor; maxdim=128, cutoff=1e-8, tol=1e-8, eager=true, krylovdim=40)
    Γ_transfer = transfer_matrix(Γ)
    λ_transfer = transfer_matrix(λ)

    X, Y, η = canonicalize_X_Y(Γ_transfer, λ_transfer; tol=tol, eager=eager, krylovdim=krylovdim)

    λ_new, U, V = update_λ(λ, X, Y)
    λ_new /= sqrt(η)

    Γ_new = update_Γ(Γ, U, V, X, Y)

    return Γ_new, λ_new
end

function update_λ(λ::ITensor, X::Matrix, Y::Matrix; maxdim=128, cutoff=1e-8)
    l21 = only(inds(λ; tags="Link,n=2,l=1"))
    l22 = only(inds(λ; tags="Link,n=2,l=2"))

    Y_tensor = ITensor(Y, prime(l21), l21)
    X_tensor = ITensor(X, l22, prime(l22))

    U, λ_new, V, _, u, v = svd(Y_tensor * λ * X_tensor, prime(l21); maxdim=maxdim, cutoff=cutoff)

    return replaceinds(λ_new, u => l21, v => l22), Matrix(U, prime(l21), u), Matrix(V, v, prime(l22))
end

function update_Γ(Γ::ITensor, U::Matrix, V::Matrix, X::Matrix, Y::Matrix)
    l02 = only(inds(Γ; tags="Link,n=0,l=2"))
    l21 = only(inds(Γ; tags="Link,n=2,l=1"))

    A = ITensor(V*inv(X), l02, prime(l02))
    B = ITensor(inv(Y)*U, prime(l21), l21)

    return A * prime(Γ) * B
end

function canonicalize_X_Y(Γ_transfer::ITensor, λ_transfer::ITensor; tol=1e-8, eager=true, krylovdim=40)
    R = Γ_transfer * λ_transfer

    l01 = Index(size(λ_transfer, 1), tags="Link,n=0,l=1")
    l02 = only(inds(Γ_transfer; tags="Link,n=0,l=2", plev=0))
    l21 = only(inds(λ_transfer; tags="Link,n=2,l=1", plev=0))
    l22 = only(inds(λ_transfer; tags="Link,n=2,l=2", plev=0))

    λ_transfer_L = replaceinds(λ_transfer, l21 => l01, prime(l21) => prime(l01), l22 => l02, prime(l22) => prime(l02))
    L = λ_transfer_L * Γ_transfer

    # wires (delta identities)
    W_R = delta(l22, l02) * delta(prime(l22), prime(l02))
    W_L = delta(l01, l21) * delta(prime(l01), prime(l21))

    # linear maps
    Rmap(X::ITensor) = W_R * (R * X)
    Lmap(X::ITensor) = (X * L) * W_L

    arn = Arnoldi(; tol, eager, krylovdim)

    # dominant right eigenvector of Rmap on (l22, prime(l22))
    v0R = randomITensor(l22, prime(l22))
    TT_R, vR, μR, infoR = schursolve(Rmap, v0R, 1, :LM, arn)
    μR = μR[1]
    VR = vR[1]

    if infoR.converged == 0
        @warn "Rmap not converged after $(infoR.numiter) iterations"
    end
    if size(TT_R, 2) > 1 && TT_R[2, 1] != 0
        @warn "Non-unique largest eigenvector of Rmap found"
    end

    # dominant right eigenvector of Lmap on (l01, prime(l01))
    v0L = randomITensor(l01, prime(l01))
    TT_L, vL, μL, infoL = schursolve(Lmap, v0L, 1, :LM, arn)
    μL = μL[1]
    VL = vL[1]

    if infoL.converged == 0
        @warn "Lmap not converged after $(infoL.numiter) iterations"
    end
    if size(TT_L, 2) > 1 && TT_L[2, 1] != 0
        @warn "Non-unique largest eigenvector of Lmap found"
    end

    @assert μL ≈ μR

    W, S, Wdag, _, _, v = svd(VR, l22)
    X = Matrix(W * sqrt.(S), l22, v)

    W, S, Wdag, _, _, v = svd(VL, l01)
    Y = Matrix(W * sqrt.(S), l01, v)

    return X, Y, μR
end

canonicalize_X_Y (generic function with 1 method)

In [98]:
Γa = ComplexF64.(reshape([1.0, 0.0], 1, 2, 1))
λa = ComplexF64.(reshape([1.0], 1, 1))
Γb = ComplexF64.(reshape([0.0, 1.0], 1, 2, 1))
λb = ComplexF64.(reshape([1.0], 1, 1))

ψ = iMPS2(Γa, λa, Γb, λb)

canonicalize(ψ)

iMPS2(ITensor ord=3
Dim 1: (dim=1|id=231|"Link,l=2,n=0")
Dim 2: (dim=2|id=167|"Site,n=1")'
Dim 3: (dim=1|id=153|"Link,l=1,n=1")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 1×2×1
[:, :, 1] =
 1.0 + 0.0im  0.0 + 0.0im
, ITensor ord=2
Dim 1: (dim=1|id=153|"Link,l=1,n=1")
Dim 2: (dim=1|id=641|"Link,l=2,n=1")
NDTensors.Diag{Float64, Vector{Float64}}
 1×1
 1.0
, ITensor ord=3
Dim 1: (dim=2|id=991|"Site,n=2")'
Dim 2: (dim=1|id=641|"Link,l=2,n=1")'
Dim 3: (dim=1|id=242|"Link,l=2,n=2")'
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2×1×1
[:, :, 1] =
 0.0 + 0.0im
 1.0 + 0.0im
, ITensor ord=2
Dim 1: (dim=1|id=164|"Link,l=1,n=2")
Dim 2: (dim=1|id=242|"Link,l=2,n=2")
NDTensors.Diag{ComplexF64, Vector{ComplexF64}}
 1×1
 1.0 - 0.0im
)

In [75]:
svd([1 0 0 0
     0 0 1 0
     0 1 0 0
     0 0 0 1])

SVD{Float64, Float64, Matrix{Float64}, Vector{Float64}}
U factor:
4×4 Matrix{Float64}:
 1.0   0.0   0.0  0.0
 0.0   0.0  -1.0  0.0
 0.0  -1.0   0.0  0.0
 0.0   0.0   0.0  1.0
singular values:
4-element Vector{Float64}:
 1.0
 1.0
 1.0
 1.0
Vt factor:
4×4 Matrix{Float64}:
  1.0   0.0   0.0   0.0
 -0.0  -1.0  -0.0  -0.0
 -0.0  -0.0  -1.0  -0.0
  0.0   0.0   0.0   1.0

In [112]:
a = Index(4, "A")
b = Index(4, "B")
U, S, V, η, u, v = svd(ITensor([1 0 0 0
     0 0 1 0
     0 1 0 0
     0 0 0 1]/4, a, b), a; maxdim=2)

ITensors.TruncSVD(ITensor ord=2
Dim 1: (dim=4|id=782|"A")
Dim 2: (dim=2|id=678|"Link,u")
NDTensors.Dense{Float64, Vector{Float64}}
 4×2
 1.0   0.0
 0.0   0.0
 0.0  -1.0
 0.0   0.0
, ITensor ord=2
Dim 1: (dim=2|id=678|"Link,u")
Dim 2: (dim=2|id=469|"Link,v")
NDTensors.Diag{Float64, Vector{Float64}}
 2×2
 0.25  0.0
 0.0   0.25
, ITensor ord=2
Dim 1: (dim=4|id=73|"B")
Dim 2: (dim=2|id=469|"Link,v")
NDTensors.Dense{Float64, Vector{Float64}}
 4×2
 1.0  -0.0
 0.0  -1.0
 0.0  -0.0
 0.0  -0.0
, Spectrum{Vector{Float64}, Float64}([0.0625, 0.0625], 0.5), (dim=2|id=678|"Link,u"), (dim=2|id=469|"Link,v"))

MethodError: MethodError: no method matching iterate(::Spectrum{Vector{Float64}, Float64})
The function `iterate` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  iterate(!Matched::ITensors.TruncSVD)
   @ ITensors ~/.julia/packages/ITensors/SOADD/src/tensor_operations/matrix_decomposition.jl:19
  iterate(!Matched::ITensors.TruncSVD, !Matched::Val{:S})
   @ ITensors ~/.julia/packages/ITensors/SOADD/src/tensor_operations/matrix_decomposition.jl:20
  iterate(!Matched::ITensors.TruncSVD, !Matched::Val{:V})
   @ ITensors ~/.julia/packages/ITensors/SOADD/src/tensor_operations/matrix_decomposition.jl:21
  ...


In [105]:
svd([1 0 0 0
     0 0 1 0
     0 1 0 0
     0 0 0 1]/4)

SVD{Float64, Float64, Matrix{Float64}, Vector{Float64}}
U factor:
4×4 Matrix{Float64}:
 1.0   0.0   0.0  0.0
 0.0   0.0  -1.0  0.0
 0.0  -1.0   0.0  0.0
 0.0   0.0   0.0  1.0
singular values:
4-element Vector{Float64}:
 0.25
 0.25
 0.25
 0.25
Vt factor:
4×4 Matrix{Float64}:
  1.0   0.0   0.0   0.0
 -0.0  -1.0  -0.0  -0.0
 -0.0  -0.0  -1.0  -0.0
  0.0   0.0   0.0   1.0

In [82]:
ten = ITensor([1 0 0 0
     0 0 1 0
     0 1 0 0
     0 0 0 1], a, b)

ITensor ord=2 (dim=4|id=533|"A") (dim=4|id=215|"B")
NDTensors.Dense{Float64, Vector{Float64}}

In [84]:
Matrix(1 ./ ten, a, b)

4×4 Matrix{Float64}:
  1.0  Inf   Inf   Inf
 Inf   Inf    1.0  Inf
 Inf    1.0  Inf   Inf
 Inf   Inf   Inf    1.0

In [52]:
Γa = ComplexF64.(reshape([1.0, 0.0], 1, 2, 1))
λa = ComplexF64.(reshape([1.0], 1, 1))
Γb = ComplexF64.(reshape([0.0, 1.0], 1, 2, 1))
λb = ComplexF64.(reshape([1.0], 1, 1))

ψ = iMPS2(Γa, λa, Γb, λb)

VR, VL = canonicalize(ψ);

In [71]:
a, b, c, d, e, f = VR

ITensors.TruncSVD(ITensor ord=2
Dim 1: (dim=1|id=479|"Link,l=2,n=2")
Dim 2: (dim=1|id=682|"Link,u")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 1×1
 -1.0 + 0.0im
, ITensor ord=2
Dim 1: (dim=1|id=682|"Link,u")
Dim 2: (dim=1|id=378|"Link,v")
NDTensors.Diag{Float64, Vector{Float64}}
 1×1
 1.0
, ITensor ord=2
Dim 1: (dim=1|id=479|"Link,l=2,n=2")'
Dim 2: (dim=1|id=378|"Link,v")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 1×1
 1.0 + 0.0im
, Spectrum{Vector{Float64}, Float64}([1.0], 0.0), (dim=1|id=682|"Link,u"), (dim=1|id=378|"Link,v"))

In [72]:
f

(dim=1|id=378|"Link,v")

In [46]:
svd(VR)

ErrorException: Must specify indices in `svd`

iMPS2(ITensor ord=3
Dim 1: (dim=1|id=717|"Link,l=2,n=0")
Dim 2: (dim=2|id=937|"Site,n=1")
Dim 3: (dim=1|id=870|"Link,l=1,n=1")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 1×2×1
[:, :, 1] =
 1.0 + 0.0im  0.0 + 0.0im
, ITensor ord=2
Dim 1: (dim=1|id=870|"Link,l=1,n=1")
Dim 2: (dim=1|id=613|"Link,l=2,n=1")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 1×1
 1.0 + 0.0im
, ITensor ord=3
Dim 1: (dim=1|id=613|"Link,l=2,n=1")
Dim 2: (dim=2|id=242|"Site,n=2")
Dim 3: (dim=1|id=406|"Link,l=1,n=2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 1×2×1
[:, :, 1] =
 0.0 + 0.0im  1.0 + 0.0im
, ITensor ord=2
Dim 1: (dim=1|id=406|"Link,l=1,n=2")
Dim 2: (dim=1|id=28|"Link,l=2,n=2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 1×1
 1.0 + 0.0im
)

In [31]:
    Γa_transfer = transfer_matrix(ψ.Γa)
    λa_transfer = transfer_matrix(ψ.λa)
    Γb_transfer = transfer_matrix(ψ.Γb)
    λb_transfer = transfer_matrix(ψ.λb)

    Γ_transfer = Γa_transfer * λa_transfer * Γb_transfer
    # Γ, λ = canonicalize_transfers(Γ_transfer, λb_transfer; maxdim=maxdim, cutoff=cutoff)

    # Γa, λa, Γb, λb = split_canonical(Γ, λ; maxdim=maxdim, cutoff=cutoff)

    # return iMPS2(Γa, λa, Γb, λb)

ITensor ord=4 (dim=1|id=717|"Link,l=2,n=0") (dim=1|id=717|"Link,l=2,n=0")' (dim=1|id=406|"Link,l=1,n=2") (dim=1|id=406|"Link,l=1,n=2")'
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

In [36]:
inds(λb_transfer; tags="Link,l=1", plev=0)

((dim=1|id=406|"Link,l=1,n=2"),)

In [23]:
struct iMPS
    Γ::Vector{ITensor}
    λ::Vector{ITensor}
    iMPS(Γ::Vector{ITensor}, λ::Vector{ITensor}) = new(Γ, λ)
end

In [24]:
function iMPS(Γ_arr::Array{ComplexF64, 3}, λ_arr::Array{ComplexF64, 2})
    χ, d, _ = size(Γ_arr)

    link0 = Index(χ, tags="Link,n=0,l=2")
    link1 = Index(χ, tags="Link,n=1,l=1")
    link2 = Index(χ, tags="Link,n=1,l=2")
    site = Index(d, tags="Site,n=1")

    Γ = ITensor(Γ_arr, (link0, site, link1))
    λ = ITensor(λ_arr, (link1, link2))

    return iMPS([Γ], [λ])
end

function iMPS(Γ_arrs::Vector{Array{ComplexF64, 3}}, λ_arrs::Vector{Array{ComplexF64, 2}})
    N = length(Γ_arrs)
    Γ = Vector{ITensor}(undef, N)
    λ = Vector{ITensor}(undef, N)

    χ, d, _ = size(Γ_arrs[1])

    links = vcat([Index(χ, tags="Link,n=0,l=2")], [Index(χ, tags="Link,n=$(i),l=$(j)") for i in 1:N, j in 1:2])
    sites = [Index(d, tags="Site,n=$(i)") for i in 1:N]

    for n in 1:N
        Γ[n] = ITensor(Γ_arrs[n], (links[2n-1], sites[n], links[2n]))
        λ[n] = ITensor(λ_arrs[n], (links[2n], links[2n+1]))
    end

    return iMPS(Γ, λ)
end

iMPS

In [15]:
transfer_matrix(tensor::ITensor) = tensor * dag(prime(tensor; tags="Link"))

transfer_matrix (generic function with 1 method)

In [25]:
iMPS(ComplexF64.(reshape([0.0, 1.0], 1, 2, 1)), ComplexF64.(reshape([1.0], 1, 1)))

iMPS(ITensor[ITensor ord=3
Dim 1: (dim=1|id=302|"Link,l=2,n=0")
Dim 2: (dim=2|id=412|"Site,n=1")
Dim 3: (dim=1|id=216|"Link,l=1,n=1")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 1×2×1
[:, :, 1] =
 0.0 + 0.0im  1.0 + 0.0im
], ITensor[ITensor ord=2
Dim 1: (dim=1|id=216|"Link,l=1,n=1")
Dim 2: (dim=1|id=3|"Link,l=2,n=1")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 1×1
 1.0 + 0.0im
])

In [ ]:
function canonicalize(ψ::iMPS)
    Γ_transfers = transfer_matrix.(ψ.Γ)
    λ_transfers = transfer_matrix.(ψ.λ)

    Γ = reduce(*, vcat(Γ_transfers, λ_transfers[1:end-1]))
    λ = λ[end]

    Γ, λ = canonicalize(Γ, λ)
    
end

In [ ]:
function right_transfer_matrix()

In [14]:
function canonicalize!(Γ::ITensor, λ::ITensor)
    Γdag = prime(dag(Γ); tags="Link")

    Γ0 = only(inds(Γ; tags="Link,n=0"))
    λ1 = only(inds(λ; tags="Link,n=1,l=1"))
    λ2 = only(inds(λ; tags="Link,n=1,l=2"))

    λR = replaceinds(λ, λ2 => prime(λ1))
    λL = replaceinds(λ, λ2 => Γ0, λ1 => prime(Γ0))

    R = Γ*Γdag*λR
    L = Γ*Γdag*λL

    return R, L
end

canonicalize! (generic function with 1 method)

In [7]:
χ = 4
N = 3
d = 2

links = vcat([Index(χ, tags="Link,n=0,l=2")], [Index(χ, tags="Link,n=$(i),l=$(j)") for i in 1:N for j in 1:2])
sites = [Index(d, tags="Site,n=$(i)") for i in 1:N]

3-element Vector{Index{Int64}}:
 (dim=2|id=656|"Site,n=1")
 (dim=2|id=203|"Site,n=2")
 (dim=2|id=653|"Site,n=3")

In [8]:
links

7-element Vector{Index{Int64}}:
 (dim=4|id=754|"Link,l=2,n=0")
 (dim=4|id=743|"Link,l=1,n=1")
 (dim=4|id=993|"Link,l=2,n=1")
 (dim=4|id=13|"Link,l=1,n=2")
 (dim=4|id=712|"Link,l=2,n=2")
 (dim=4|id=711|"Link,l=1,n=3")
 (dim=4|id=827|"Link,l=2,n=3")

Base.Meta.ParseError: ParseError:
# Error @ /Users/jhauser/Code/U1_SWSSB/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W1sZmlsZQ==.jl:1:47
function canonicalize!(Γ::ITensor, λ::ITensor)
#                                             └ ── premature end of input

In [7]:
Index(2, tags="Site")

(dim=2|id=142|"Site")